# Multi-Agent Chat System

A 3-way conversation system with AI agents having distinct personalities:

1. **Andrew** - Argumentative and cynical, challenges everything
2. **Bill** - Analytical and objective, provides facts and logical corrections
3. **Christy** - Polite and courteous, tries to find common ground

## Features
- Multi-agent conversation with distinct personalities
- Different models for different agents (Gemma4:e2b for Andrew/Bill, Llama3.2:3b for Christy)
- Temperature tuning per agent personality
- Gradio interface for interactive conversations
- **Real-time streaming responses** - Watch agents respond word-by-word

## Setup and Configuration

In [ ]:
# Import dependencies
import os
import warnings
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

# Suppress Starlette deprecation warning (library-level issue)
warnings.filterwarnings("ignore", message=".*HTTP_422_UNPROCESSABLE_ENTITY.*", category=DeprecationWarning)

# Load environment variables
load_dotenv(override=True)

# Configuration
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1')

# Model configurations
MODEL_ANDREW_BILL = "gemma4:e2b"  # For Andrew and Bill
MODEL_CHRISTY = "llama3.2:3b"     # For Christy

# Initialize OpenAI client for Ollama
ollama = OpenAI(api_key="ollama", base_url=OLLAMA_BASE_URL)

print("Configuration:")
print(f"   Ollama Base URL: {OLLAMA_BASE_URL}")
print(f"   Andrew/Bill Model: {MODEL_ANDREW_BILL}")
print(f"   Christy Model: {MODEL_CHRISTY}")

## System Prompts for Each Agent

In [ ]:
# System prompts for each agent
andrew_system = """You are Andrew, a chatbot who is very argumentative. 
You disagree with anything said in the conversation and challenge everything in a snarky, cynical way.
You are in a 3-way conversation with Bill and Christy. 
CRITICAL: Write exactly 1 short sentence. Do not write paragraphs."""

bill_system = """You are Bill, a highly analytical, objective chatbot. 
You don't take sides emotionally; you just jump in with data, facts, or logical corrections.
You are in a 3-way conversation with Andrew and Christy.
CRITICAL: Write exactly 1 short sentence. Do not write paragraphs."""

christy_system = """You are Christy, a very polite, courteous chatbot. 
You try to find common ground, calm down any arguments, and keep the conversation friendly.
You are in a 3-way conversation with Andrew and Bill.
CRITICAL: Write exactly 1 sentence. Do not write paragraphs."""

print("System prompts defined for Andrew, Bill, and Christy")

## Agent Caller Function

In [ ]:
def call_agent(agent_name, model_name, system_prompt, temp, conversation_history):
    """
    Call an agent with the given parameters and conversation history.
    Returns a streaming response generator.
    """
    formatted_conversation = "\n".join(conversation_history)
    
    user_prompt = f"""You are {agent_name}, in a conversation with the other two agents.
The conversation history so far is:
{formatted_conversation}

Now, respond with what you would like to say next as {agent_name}. 
Do not prefix your answer with your name, just type your response directly."""

    response = ollama.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=temp,
        stream=True
    )
    return response

## Gradio Interface with Streaming

In [ ]:
def chat_interface(initial_topic, num_rounds):
    """
    Gradio interface function that runs the conversation with streaming support.
    Yields accumulated conversation for real-time display.
    """
    # Validate input
    if not initial_topic.strip():
        yield "Please provide an initial topic for Andrew to start the conversation."
        return
    
    try:
        num_rounds = int(num_rounds)
        if num_rounds < 1:
            yield "Number of rounds must be at least 1."
            return
    except ValueError:
        yield "Please provide a valid number for rounds."
        return
    
    # Initialize conversation with Andrew's opening statement
    conversation_history = [f"Andrew: {initial_topic}"]
    
    # Build and yield the initial conversation
    full_conversation = f"### 🤖 **Andrew:**\n{initial_topic}\n\n"
    yield full_conversation
    
    # Run conversation rounds
    for _ in range(num_rounds):
        # 1. BILL RESPONDS
        bill_response = ""
        bill_stream = call_agent("Bill", MODEL_ANDREW_BILL, bill_system, temp=0.3, conversation_history=conversation_history)
        
        # Stream Bill's response
        full_conversation += "### 📊 **Bill:**\n"
        yield full_conversation
        
        for chunk in bill_stream:
            content = chunk.choices[0].delta.content or ''
            bill_response += content
            full_conversation += content
            yield full_conversation
        
        bill_reply = bill_response.strip()
        conversation_history.append(f"Bill: {bill_reply}")
        full_conversation += "\n\n"
        yield full_conversation
        
        # 2. CHRISTY RESPONDS
        christy_response = ""
        christy_stream = call_agent("Christy", MODEL_CHRISTY, christy_system, temp=0.5, conversation_history=conversation_history)
        
        # Stream Christy's response
        full_conversation += "### 🦊 **Christy:**\n"
        yield full_conversation
        
        for chunk in christy_stream:
            content = chunk.choices[0].delta.content or ''
            christy_response += content
            full_conversation += content
            yield full_conversation
        
        christy_reply = christy_response.strip()
        conversation_history.append(f"Christy: {christy_reply}")
        full_conversation += "\n\n"
        yield full_conversation
        
        # 3. ANDREW RESPONDS
        andrew_response = ""
        andrew_stream = call_agent("Andrew", MODEL_ANDREW_BILL, andrew_system, temp=0.8, conversation_history=conversation_history)
        
        # Stream Andrew's response
        full_conversation += "### 🤖 **Andrew:**\n"
        yield full_conversation
        
        for chunk in andrew_stream:
            content = chunk.choices[0].delta.content or ''
            andrew_response += content
            full_conversation += content
            yield full_conversation
        
        andrew_reply = andrew_response.strip()
        conversation_history.append(f"Andrew: {andrew_reply}")
        full_conversation += "\n\n"
        yield full_conversation

# Create Gradio interface with streaming
with gr.Blocks(title="Multi-Agent Chat System") as demo:
    gr.Markdown("# 🤖 Multi-Agent Chat System")
    gr.Markdown("""
    Watch Andrew (argumentative), Bill (analytical), and Christy (polite) engage in a 3-way conversation!
    
    - **Andrew**: Uses Gemma4:e2b with high temperature (0.8) - argumentative and cynical
    - **Bill**: Uses Gemma4:e2b with low temperature (0.3) - analytical and factual
    - **Christy**: Uses Llama3.2:3b with medium temperature (0.5) - polite and diplomatic
    """)
    
    with gr.Row():
        with gr.Column():
            initial_topic = gr.Textbox(
                label="Initial Topic",
                placeholder="Andrew's opening statement...",
                value="I think all programming languages except Assembly are completely lazy and pointless."
            )
            num_rounds = gr.Slider(
                minimum=1,
                maximum=10,
                value=3,
                step=1,
                label="Number of Conversation Rounds"
            )
            submit_btn = gr.Button("Start Conversation", variant="primary")
        
        with gr.Column():
            output = gr.Markdown(label="Conversation")
    
    submit_btn.click(
        fn=chat_interface,
        inputs=[initial_topic, num_rounds],
        outputs=output
    )
    
    gr.Examples(
        examples=[
            ["I think all programming languages except Assembly are completely lazy and pointless.", 3],
            ["Coffee is absolutely the worst beverage ever created.", 2],
            ["Remote work is destroying company culture and productivity.", 4],
            ["Artificial intelligence will never replace human creativity.", 3],
        ],
        inputs=[initial_topic, num_rounds],
    )

print("Gradio interface created successfully with streaming support!")

## Launch the Interface

In [ ]:
# Launch the Gradio interface
if __name__ == "__main__":
    demo.launch(share=False)

## Technical Implementation Details

### Agent Personalities
- **Andrew**: High temperature (0.8) for more creative, argumentative responses
- **Bill**: Low temperature (0.3) for consistent, factual responses
- **Christy**: Medium temperature (0.5) for balanced, diplomatic responses

### Model Selection
- **Andrew & Bill**: Gemma4:e2b - Good for creative and analytical tasks
- **Christy**: Llama3.2:3b - Efficient model for polite, conversational responses

### Conversation Flow
1. Andrew starts with the initial topic
2. Each round: Bill → Christy → Andrew
3. Full conversation history is maintained and passed to each agent
4. Responses are limited to 1 sentence per agent for focused dialogue

### Streaming Implementation
- **Real-time streaming**: Watch agents respond word-by-word as they generate text
- **Accumulated response**: Each yield returns the complete conversation so far
- **Agent-level streaming**: Each agent's response streams individually for dramatic effect
- **Null handling**: Uses `or ''` pattern to handle None content gracefully

### Warning Suppression
- The Starlette deprecation warning is a library-level compatibility issue
- It's suppressed using `warnings.filterwarnings()` for cleaner output
- This doesn't affect functionality, only console output

### Future Improvements
- Add conversation memory across sessions
- Implement more personality options
- Support for custom system prompts
- Conversation export functionality
- Add voice synthesis for each agent
- Visual avatars for each agent